In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

DATA_DIR = Path(r"../datasets")
PROCESSED_DIR = DATA_DIR / "processed"

train = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
movies = pd.read_csv(PROCESSED_DIR / "phase2_movies.csv")
weighted_matrix = load_npz(PROCESSED_DIR / "weighted_matrix.npz")

user_ids = np.load(PROCESSED_DIR / "cf_user_ids.npy")
movie_ids = np.load(PROCESSED_DIR / "cf_movie_ids.npy")
user_factors = np.load(PROCESSED_DIR / "cf_user_factors.npy")
movie_factors = np.load(PROCESSED_DIR / "cf_movie_factors.npy")

user_to_index = pd.Series(np.arange(len(user_ids)), index=user_ids)
movie_id_to_index = pd.Series(movies.index.to_numpy(), index=movies["movieId"])

print("Content:", weighted_matrix.shape)
print("CF users:", user_factors.shape)
print("CF movies:", movie_factors.shape)


In [ ]:
## Build content score
from scipy.sparse import csr_matrix

weighted_matrix = csr_matrix(weighted_matrix)

print("Weighted matrix:", weighted_matrix.shape)
print("Type:", type(weighted_matrix))
POSITIVE_THRESHOLD = 4.0

def build_user_vector(user_id):

    history = train[
        (train["userId"] == user_id) &
        (train["rating"] >= POSITIVE_THRESHOLD)
    ].copy()

    history["matrix_index"] = history["movieId"].map(
        movie_id_to_index
    )

    history = history.dropna(
        subset=["matrix_index"]
    )

    if history.empty:
        return None

    indices = (
        history["matrix_index"]
        .astype(int)
        .to_numpy()
    )

    vectors = weighted_matrix[indices]

    weights = (
        history["rating"]
        .to_numpy(dtype=float) - 3.0
    )

    user_vector = (
        vectors.multiply(weights[:, None])
        .sum(axis=0)
        / weights.sum()
    )

    return csr_matrix(user_vector)

def content_scores(user_id):

    vector = build_user_vector(user_id)

    if vector is None:
        return None

    return cosine_similarity(
        vector,
        weighted_matrix
    ).ravel()


In [17]:
## Build collaborative score

def collaborative_scores(user_id):
    if user_id not in user_to_index.index:
        return None

    uidx = int(user_to_index[user_id])
    cf_scores = user_factors[uidx] @ movie_factors.T

    result = pd.Series(
        cf_scores,
        index=movie_ids,
        dtype=float
    )

    return result


In [18]:
## Normalize signals
def minmax(values):
    values = np.asarray(values, dtype=float)
    minimum = np.nanmin(values)
    maximum = np.nanmax(values)

    if maximum == minimum:
        return np.zeros_like(values)

    return (values - minimum) / (maximum - minimum)


In [19]:

## Create the hybrid score

CONTENT_WEIGHT = 0.50
COLLAB_WEIGHT = 0.50

def hybrid_recommend(user_id, n=10):
    c_scores = content_scores(user_id)

    if c_scores is None:
        return pd.DataFrame()

    result = movies.copy()
    result["content_score"] = minmax(c_scores)

    cf = collaborative_scores(user_id)

    if cf is None:
        result["collaborative_score"] = 0.0
    else:
        result["collaborative_score"] = (
            result["movieId"]
            .map(cf)
            .fillna(0.0)
        )

        result["collaborative_score"] = minmax(
            result["collaborative_score"]
        )

    result["hybrid_score"] = (
        CONTENT_WEIGHT * result["content_score"] +
        COLLAB_WEIGHT * result["collaborative_score"]
    )

    rated_ids = set(
        train.loc[
            train["userId"] == user_id,
            "movieId"
        ]
    )

    result = result[
        ~result["movieId"].isin(rated_ids)
    ]

    return (
        result
        .sort_values("hybrid_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


In [ ]:
test_user = int(train["userId"].iloc[0])
display(hybrid_recommend(test_user, 10))


In [ ]:
# Build a simple popularity fallback from training ratings.

popularity = (
    train.groupby("movieId")["rating"]
    .agg(
        rating_count="count",
        average_rating="mean"
    )
    .reset_index()
)

MIN_COUNT = 20

popular_movies = (
    popularity[popularity["rating_count"] >= MIN_COUNT]
    .sort_values(
        ["average_rating", "rating_count"],
        ascending=False
    )
)

display(popular_movies.head(10))


In [22]:
## Phase 6 validation
assert CONTENT_WEIGHT + COLLAB_WEIGHT == 1.0

hybrid_result = hybrid_recommend(test_user, 10)

if not hybrid_result.empty:
    rated = set(
        train.loc[
            train["userId"] == test_user,
            "movieId"
        ]
    )
    assert not set(hybrid_result["movieId"]).intersection(rated)

print("PHASE 6 HYBRID MODEL VALIDATION PASSED")


PHASE 6 HYBRID MODEL VALIDATION PASSED
